## Feature Engineering

> **Construction of the model-ready feature dataset for NBA win prediction.**  
> Features are built from historical team context only, with explicit safeguards against data leakage.  
> **Input:** `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv`  
> **Output:** `data/3_features/features_nba_data_2000-01_2025-26.csv`.

---

**Pipeline**

1. Setup — configuration, dataset versioning, paths, and feature parameters
2. Feature Plan — source-to-feature mapping and modeling rationale
3. Per-Team View — chronological team-game representation
4. Rolling Features — prior-game moving averages with `shift(1)`
5. Context Features — streak, rest days, and back-to-back indicators
6. Ratio Features — shooting-efficiency measures
7. Normalizations — within-season z-scores and historical trend encoding
8. Game View Reconstruction — home/away joins and previous-season win rate
9. Differential Features — home-minus-away comparisons and target definition
10. Anomalous Season Flag — COVID bubble context preservation
11. Filtering — minimum-history enforcement
12. Temporal Train / Validation / Test Split — forecasting-style evaluation design
13. Saving — model-ready feature artifact persistence

### 1. Setup

* **What:** Imports data-processing libraries, loads project configuration, defines input/output paths, and sets feature-engineering parameters.
* **Why:** To centralize the dataset version, rolling-window size, and minimum-history threshold before transformations begin.
* **Reasoning/Choices:** Configuration-driven paths and explicit constants make the generated feature dataset reproducible across season ranges and prevent hidden modeling assumptions.

In [ ]:
import logging
import sys
from pathlib import Path

# Set the root directory and update the system path
ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import polars as pl

from src.loader import load_config

# Load configuration settings
config = load_config()

# Configure logging settings
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

# Initialize notebook versioning
NOTEBOOK_VERSION = "1.0.1"
log.info(f"Feature Engineering notebook v{NOTEBOOK_VERSION} — initialized")

In [ ]:
# Retrieve season range from configuration
start_year  = config["settings"]["start_season"]
end_year    = config["settings"]["end_season"]

# Format season strings (e.g., "2025-26")
start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"
DATASET_VERSION = f"{start_season}_{end_season}"

# Define directory paths for processed and feature data
PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]
FEATURES_DIR  = ROOT / "data" / "3_features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Define input and output file paths
IN_FILE  = PROCESSED_DIR / f"cleaned_nba_data_{DATASET_VERSION}.csv"
OUT_FILE = FEATURES_DIR  / f"features_nba_data_{DATASET_VERSION}.csv"

# Set parameters for rolling window calculations
WINDOW_N           = 10
MIN_GAMES_REQUIRED = 5

# Log execution parameters
log.info(f"Input              : {IN_FILE}")
log.info(f"Output             : {OUT_FILE}")
log.info(f"Rolling window N   : {WINDOW_N}")
log.info(f"Min games required : {MIN_GAMES_REQUIRED}")

In [ ]:
# Load the dataset from CSV
df = pl.read_csv(IN_FILE, try_parse_dates=True)

# Log dataset dimensions and column names
log.info(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
log.info(f"Columns: {df.columns}")

# Display the first 3 rows
df.head(3)

### 2. Feature Plan

* **What:** Documents the transformation plan from cleaned team-game variables to model-ready features.
* **Why:** To make every engineered feature traceable to a raw input, basketball concept, and predictive purpose.
* **Reasoning/Choices:** The plan separates recent form, efficiency, context, normalization, and prior-season strength so the final model input space remains interpretable.

#### 2.1 Column Mapping

* **What:** Maps each cleaned input column to its transformation and final feature name.
* **Why:** To document how raw basketball measures become leakage-safe model inputs.
* **Reasoning/Choices:** Rolling features use prior games only, ratio features capture efficiency, and normalization features reduce era-driven distribution drift.

| Raw Column | Transformation | Final Feature | Notes |
| --- | --- | --- | --- |
| `net_rating` | `rolling_mean(N)`, `shift(1)` per team | `net_rating_rolling_N` | Strongest single predictor (EDA) |
| `pts` | `rolling_mean(N)`, `shift(1)` per team | `avg_pts_last_N` | Recent offensive trend |
| `ts_pct` | `rolling_mean(N)`, `shift(1)` per team | `avg_ts_pct_last_N` | Shooting efficiency last N games |
| `pace` | `rolling_mean(N)`, `shift(1)` per team | `avg_pace_last_N` | Recent game pace |
| `wl` | win indicator + `rolling_mean(N)`, `shift(1)` | `winrate_last_N` | Recent form (win rate last N) |
| `wl` | count consecutive streaks, `shift(1)` | `streak` | Momentum (signed int: +win, −loss) |
| `game_date` | `diff()` per team | `rest_days` | Rest days since previous game |
| `rest_days` | `== 1` | `is_back_to_back` | Back-to-back flag (bool) |
| `fgm / fga` | division; `fga == 0 → null` | `fg_pct` | Field goal efficiency |
| `fg3m / fg3a` | division; `fg3a == 0 → null` | `fg3_pct` | Three-point efficiency |
| `ftm / fta` | division; `fta == 0 → null` | `ft_pct` | Free throw efficiency |
| `ts_pct` | z-score within-season | `ts_pct_zscore` | Normalize structural drift |
| `pace` | z-score within-season | `pace_zscore` | Normalize structural drift |
| `season` | ordinal index z-norm | `season_zscore` | Modeling historical trend |
| `wl` (prev. season) | mean wins per (team, season−1) | `win_rate_prev_season` | Inter-seasonal stability |

### 3. Per-Team View

* **What:** Creates one chronological row per `(team_id, game_id)` with the variables needed for feature construction.
* **Why:** To build team-history features from the correct temporal perspective.
* **Reasoning/Choices:** Sorting by `team_id` and `game_date` is required before rolling calculations; without this ordering, recent-form features could mix future and past games.

In [ ]:
# Define the columns to retain for analysis
KEEP_COLS = [
    "game_id", "game_date", "season", "team_id", "team_abbreviation",
    "home_away", "wl", "pts", "plus_minus",
    "net_rating", "ts_pct", "pace",
    "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
]

# Create a team-oriented view sorted by team and date
team_view = df.select(KEEP_COLS).sort(["team_id", "game_date"])

# Log summary statistics of the filtered view
log.info(f"Per-team view: {team_view.shape[0]:,} rows × {team_view.shape[1]} columns")
log.info(f"Teams: {team_view['team_id'].n_unique()} | Seasons: {team_view['season'].n_unique()}")
log.info(f"Date range: {team_view['game_date'].min()} → {team_view['game_date'].max()}")

# Display the first 3 rows
team_view.head(3)

### 4. Rolling Features (`shift(1)` — No Data Leakage)

* **What:** Computes rolling averages for recent team performance using only games before the current matchup.
* **Why:** To represent current form without leaking information from the game being predicted.
* **Reasoning/Choices:** `shift(1)` enforces temporal causality, while `MIN_GAMES_REQUIRED` prevents unstable early-season estimates from entering the model.

In [ ]:
# Create rolling features for key metrics to capture recent team performance
team_view = team_view.with_columns([
    # Calculate rolling mean of net rating, shifted by 1 to avoid data leakage
    pl.col("net_rating")
        .rolling_mean(window_size=WINDOW_N, min_samples=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"net_rating_rolling_{WINDOW_N}"),

    # Calculate rolling mean of points
    pl.col("pts").cast(pl.Float64)
        .rolling_mean(window_size=WINDOW_N, min_samples=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"avg_pts_last_{WINDOW_N}"),

    # Calculate rolling mean of True Shooting percentage
    pl.col("ts_pct")
        .rolling_mean(window_size=WINDOW_N, min_samples=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"avg_ts_pct_last_{WINDOW_N}"),

    # Calculate rolling mean of pace
    pl.col("pace")
        .rolling_mean(window_size=WINDOW_N, min_samples=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"avg_pace_last_{WINDOW_N}"),

    # Calculate rolling win rate
    (pl.col("wl") == "W").cast(pl.Float64)
        .rolling_mean(window_size=WINDOW_N, min_samples=MIN_GAMES_REQUIRED)
        .shift(1)
        .over("team_id")
        .alias(f"winrate_last_{WINDOW_N}"),
])

# Log null counts for each generated rolling feature
rolling_cols = [
    f"net_rating_rolling_{WINDOW_N}", f"avg_pts_last_{WINDOW_N}",
    f"avg_ts_pct_last_{WINDOW_N}", f"avg_pace_last_{WINDOW_N}",
    f"winrate_last_{WINDOW_N}",
]
for c in rolling_cols:
    log.info(f"{c}: null={team_view[c].null_count():,}")

### 5. Context Features

* **What:** Builds contextual variables such as streak, rest days, and back-to-back status.
* **Why:** To capture schedule pressure and momentum effects that are not fully represented by box-score efficiency metrics.
* **Reasoning/Choices:** These features are calculated from prior chronological records so they describe pre-game context rather than post-game outcomes.

In [ ]:
def _streak_per_team(wl_vals: np.ndarray) -> np.ndarray:
    """Signed streak at each position using only PREVIOUS games (no leakage).

    Positive = consecutive wins ending before this game.
    Negative = consecutive losses ending before this game.
    Position 0 → streak = 0 (no prior history).
    """
    n = len(wl_vals)
    streaks = np.zeros(n, dtype=np.int32)
    # Calculate streak by looking back at previous game results
    for i in range(1, n):
        prev = wl_vals[i - 1]
        count = 1
        j = i - 2
        while j >= 0 and wl_vals[j] == prev:
            count += 1
            j -= 1
        streaks[i] = int(count) if prev == "W" else -int(count)
    return streaks


# Compute per-team streak in pandas (concise; ~30 teams × ~2,000 games each)
_key = ["game_id", "team_id"]
_tmp = (
    team_view.select(_key + ["game_date", "wl"])
    .to_pandas()
    .sort_values(["team_id", "game_date"])
    .reset_index(drop=True)
)
_tmp["streak"] = 0

# Apply the streak calculation per team group
for _, grp in _tmp.groupby("team_id", sort=True):
    _tmp.loc[grp.index, "streak"] = _streak_per_team(grp["wl"].values)

# Convert back to Polars and join with the main DataFrame
streak_pl = (
    pl.from_pandas(_tmp[_key + ["streak"]])
    .with_columns(pl.col("team_id").cast(team_view["team_id"].dtype))
)
team_view = team_view.join(streak_pl, on=_key, how="left")

# Log streak distribution and null counts
log.info(
    f"streak: min={team_view['streak'].min()}, max={team_view['streak'].max()}, "
    f"null={team_view['streak'].null_count()}"
)

In [ ]:
# Calculate rest days between games for each team
team_view = (
    team_view
    .with_columns([
        (
            pl.col("game_date")
            - pl.col("game_date").shift(1).over("team_id")
        ).dt.total_days().alias("rest_days")
    ])
    # Identify back-to-back games (1 day of rest)
    .with_columns([
        (pl.col("rest_days") == 1).fill_null(False).alias("is_back_to_back")
    ])
)

# Log summary of rest days and back-to-back game frequency
b2b_n = int(team_view.filter(pl.col("is_back_to_back"))["game_id"].len())
log.info(f"rest_days       : null={team_view['rest_days'].null_count():,} (first game per team)")
log.info(f"is_back_to_back : {b2b_n:,} games ({b2b_n / team_view.height:.1%})")

### 6. Ratio Features

* **What:** Derives shooting efficiency ratios for field goals, three-pointers, and free throws.
* **Why:** To express scoring efficiency independently from raw attempt volume.
* **Reasoning/Choices:** Division-by-zero cases are set to null because a missing opportunity is not the same as zero efficiency; later filtering or preprocessing can handle those values explicitly.

Calculated from the cleaned raw data (not the rolling data).
Any denominator where `== 0` is set to `null` — not `inf` — to prevent distortion of downstream statistics and gradient-boosted models.

In [ ]:
# Calculate shooting percentages, handling division by zero by assigning None
team_view = team_view.with_columns([
    pl.when(pl.col("fga") > 0)
      .then(pl.col("fgm").cast(pl.Float64) / pl.col("fga").cast(pl.Float64))
      .otherwise(None)
      .alias("fg_pct"),

    pl.when(pl.col("fg3a") > 0)
      .then(pl.col("fg3m").cast(pl.Float64) / pl.col("fg3a").cast(pl.Float64))
      .otherwise(None)
      .alias("fg3_pct"),

    pl.when(pl.col("fta") > 0)
      .then(pl.col("ftm").cast(pl.Float64) / pl.col("fta").cast(pl.Float64))
      .otherwise(None)
      .alias("ft_pct"),
])

# Log the count of null values resulting from zero-attempt games
for col in ["fg_pct", "fg3_pct", "ft_pct"]:
    null_n = team_view[col].null_count()
    log.info(f"{col}: null={null_n} (zero-denominator games → null, not inf)")

### 7. Normalizations

* **What:** Applies within-season z-score normalization to selected performance and temporal trend variables.
* **Why:** To reduce era effects caused by changes in pace, offensive efficiency, and league context over time.
* **Reasoning/Choices:** Season-relative normalization lets models compare teams within the competitive environment they actually played in, rather than against a pooled multi-decade scale.

* `ts_pct_zscore` & `pace_zscore`: *Within-season* z-score calculation to account for the structural drift of the NBA era. Since the absolute values of `pace` and `ts_pct` have increased significantly over the last 25 years, comparing raw values directly would introduce a systematic bias into the model.
* **`season_zscore`**: Z-normalization applied to the numerical season index. This allows the model to learn longitudinal trends (e.g., the surge in three-point shooting, the increasing pace) without being sensitive to the scale of the specific year.

In [ ]:
# Calculate seasonal statistics for normalization
season_stats = (
    team_view
    .group_by("season")
    .agg([
        pl.col("ts_pct").mean().alias("ts_pct_mean"),
        pl.col("ts_pct").std(ddof=1).alias("ts_pct_std"),
        pl.col("pace").mean().alias("pace_mean"),
        pl.col("pace").std(ddof=1).alias("pace_std"),
    ])
)

# Join seasonal stats and calculate Z-scores for TS% and Pace
team_view = (
    team_view
    .join(season_stats, on="season", how="left")
    .with_columns([
        ((pl.col("ts_pct") - pl.col("ts_pct_mean")) / pl.col("ts_pct_std"))
            .alias("ts_pct_zscore"),
        ((pl.col("pace") - pl.col("pace_mean")) / pl.col("pace_std"))
            .alias("pace_zscore"),
    ])
    .drop(["ts_pct_mean", "ts_pct_std", "pace_mean", "pace_std"])
)

# Extract season year index and apply Z-normalization
team_view = team_view.with_columns(
    pl.col("season").str.slice(0, 4).cast(pl.Int32).alias("season_year")
)
sy_mean = team_view["season_year"].mean()
sy_std  = team_view["season_year"].std(ddof=1)
team_view = team_view.with_columns(
    ((pl.col("season_year") - sy_mean) / sy_std).alias("season_zscore")
)

# Log normalization verification metrics
log.info(f"ts_pct_zscore : mean={team_view['ts_pct_zscore'].mean():.4f}, std={team_view['ts_pct_zscore'].std():.4f}")
log.info(f"pace_zscore   : mean={team_view['pace_zscore'].mean():.4f}, std={team_view['pace_zscore'].std():.4f}")
log.info(f"season_zscore : [{team_view['season_zscore'].min():.2f}, {team_view['season_zscore'].max():.2f}]")

### 8. Game View Reconstruction

* **What:** Converts team-level rows into matchup-level records with separate home and away feature sets.
* **Why:** To align the dataset with the prediction task: one row per game, predicting whether the home team wins.
* **Reasoning/Choices:** Reconstructing games after team-level history is computed keeps rolling statistics leakage-safe while producing model inputs that directly compare opponents.

#### 8.1 Previous Season Win Rate by Team

* **What:** Computes each team's win rate from the previous season and attaches it to the current season.
* **Why:** To represent medium-term team strength before enough current-season games are available.
* **Reasoning/Choices:** Prior-season win rate is lagged by construction, so it adds historical context without using information from the target game.

In [ ]:
# Calculate win rate per team for each season
win_rate_season = (
    team_view
    .group_by(["team_id", "season_year"])
    .agg((pl.col("wl") == "W").mean().alias("win_rate_season"))
)

# Prepare a lookup table to associate each team's performance with the following season
# This maps the win rate from season S to be joined onto season S+1
win_rate_prev = (
    win_rate_season
    .with_columns((pl.col("season_year") + 1).alias("lookup_year"))
    .rename({"win_rate_season": "win_rate_prev_season"})
    .select(["team_id", "lookup_year", "win_rate_prev_season"])
)

# Join the previous season's win rate into the main team view
team_view = team_view.join(
    win_rate_prev,
    left_on=["team_id", "season_year"],
    right_on=["team_id", "lookup_year"],
    how="left",
)

# Log null counts for teams without historical data (e.g., expansion teams)
null_wrps = team_view["win_rate_prev_season"].null_count()
log.info(
    f"win_rate_prev_season: null={null_wrps:,} "
    f"(teams in their first dataset season or expansion teams after 2000-01)"
)

#### 8.2 Join Home and Away into a Game-Level View

* **What:** Joins the home-team and away-team records for the same `game_id` into a single matchup row.
* **Why:** To expose both opponents' pre-game features side by side for supervised learning.
* **Reasoning/Choices:** Prefixing columns by venue keeps feature meaning explicit and avoids ambiguity when the same metric exists for both teams.

Each row of the `game_view` represents a single game with the features of both teams side-by-side (`home_*` and `away_*`). `season_zscore` is a game-level feature (identical for both teams) and is included only once from the home side.

In [ ]:
# Define feature and metadata column groupings for the joined game view
FEATURE_COLS = [
    f"net_rating_rolling_{WINDOW_N}",
    f"avg_pts_last_{WINDOW_N}",
    f"avg_ts_pct_last_{WINDOW_N}",
    f"avg_pace_last_{WINDOW_N}",
    f"winrate_last_{WINDOW_N}",
    "streak",
    "rest_days",
    "is_back_to_back",
    "fg_pct",
    "fg3_pct",
    "ft_pct",
    "ts_pct_zscore",
    "pace_zscore",
    "win_rate_prev_season",
]

TARGET_RAW   = ["wl", "pts"]
META_COLS    = ["game_id", "game_date", "season", "season_zscore"]
TEAM_ID_COLS = ["team_id", "team_abbreviation"]

# Filter for home teams and prefix columns accordingly
home_df = (
    team_view.filter(pl.col("home_away") == "Home")
    .select(META_COLS + TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS)
    .rename({c: f"home_{c}" for c in TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS})
)

# Filter for away teams and prefix columns accordingly
away_df = (
    team_view.filter(pl.col("home_away") == "Away")
    .select(["game_id"] + TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS)
    .rename({c: f"away_{c}" for c in TEAM_ID_COLS + TARGET_RAW + FEATURE_COLS})
)

# Join home and away dataframes on game_id to create a single row per game
game_view = home_df.join(away_df, on="game_id", how="inner")

# Log summary of the final game-level dataset
log.info(f"Game view: {game_view.shape[0]:,} rows × {game_view.shape[1]} columns")
log.info(f"home_* columns (first 5): {[c for c in game_view.columns if c.startswith('home_')][:5]}")

# Display the first 2 rows
game_view.head(2)

### 9. Differential Features

* **What:** Creates home-minus-away feature differences and defines the classification and regression targets.
* **Why:** To represent matchup advantage directly instead of forcing models to infer every comparison from duplicated home and away columns.
* **Reasoning/Choices:** Differential features are interpretable basketball measures: positive values indicate a home-team edge in recent form, efficiency, rest, or historical strength.

For each `home_X` / `away_X` pair, `X_diff = home_X − away_X` is calculated.
The differentials capture the *relative advantage* of the home team over the visiting team.

**Target**:

* **`home_win`** (Int8, binary): 1 if the home team won.
* **`point_differential`** (Float64): `home_pts − away_pts`.

In [ ]:
# Separate features into numeric and boolean categories for differentiation
NUMERIC_FEAT = [c for c in FEATURE_COLS if c != "is_back_to_back"]
BOOL_FEAT    = ["is_back_to_back"]

# Construct expressions to calculate the difference between home and away team metrics
diff_exprs = (
    [
        (pl.col(f"home_{c}") - pl.col(f"away_{c}")).alias(f"{c}_diff")
        for c in NUMERIC_FEAT
    ]
    + [
        (
            pl.col(f"home_{c}").cast(pl.Int8) - pl.col(f"away_{c}").cast(pl.Int8)
        ).alias(f"{c}_diff")
        for c in BOOL_FEAT
    ]
)

# Apply difference calculations and derive target variables (win binary, point spread)
game_view = game_view.with_columns(
    diff_exprs
    + [
        (pl.col("home_wl") == "W").cast(pl.Int8).alias("home_win"),
        (
            pl.col("home_pts").cast(pl.Float64) - pl.col("away_pts").cast(pl.Float64)
        ).alias("point_differential"),
    ]
)

# Log summary statistics for the target variables
home_win_rate = float(game_view["home_win"].mean())
log.info(f"Differential features calculated : {len(diff_exprs)}")
log.info(
    f"Target home_win: {game_view['home_win'].sum():,} home wins / "
    f"{game_view.height:,} games ({home_win_rate:.3f})"
)
log.info(
    f"Target point_differential: "
    f"mean={game_view['point_differential'].mean():.2f}, "
    f"std={game_view['point_differential'].std():.2f}"
)

### 10. Anomalous Season Flag — COVID Bubble (2019-20)

* **What:** Adds a binary indicator for games from the disrupted 2019-20 season context.
* **Why:** To let models account for the unusual schedule, venue, and competitive conditions of the COVID bubble period.
* **Reasoning/Choices:** Flagging the season preserves the observations while allowing the model to learn whether this context changes predictive relationships.

The 2019-20 season was completed in a bubble (Orlando, FL) without crowds and without traditional travel. This de facto eliminates the home advantage and distorts `pace`, `ts_pct`, and `rest_days` patterns compared to regular seasons (H4 from Bivariate EDA).

**Documented Decision**: The season is maintained in the dataset by adding the `is_covid_bubble = 1` flag. Excluding the season will be tested during modeling (comparing AUC / RMSE with `is_covid_bubble` as a feature vs. complete exclusion).

In [ ]:
# Flag games played during the 2019-20 "bubble" season to isolate its unique environment
game_view = game_view.with_columns(
    (pl.col("season") == "2019-20").cast(pl.Int8).alias("is_covid_bubble")
)

# Log the count of bubble games and note the strategic decision for model evaluation
n_bubble = int(game_view["is_covid_bubble"].sum())
log.info(f"is_covid_bubble: {n_bubble:,} games flagged (2019-20 season)")
log.info(
    "Decision: KEEP with flag. "
    "Exclusion to be tested during modeling (Hypothesis H4 from Bivariate EDA)."
)

### 11. Filtering — `MIN_GAMES_REQUIRED`

* **What:** Removes rows that do not have enough historical games to support rolling features.
* **Why:** To avoid training on unstable or null-heavy early-season records.
* **Reasoning/Choices:** The threshold trades off sample size against feature reliability; filtered rows are excluded because their recent-form estimates are not comparable to mature rolling windows.

Games are discarded if at least one of the two teams has fewer than `MIN_GAMES_REQUIRED` previous games in the dataset. These rows have `null` values in the rolling features (due to `min_periods=MIN_GAMES_REQUIRED`). `net_rating_rolling_N` is used as the sentinel column.

In [ ]:
# Identify a sentinel column (e.g., net rating) to determine data availability for rolling features
SENTINEL = f"net_rating_rolling_{WINDOW_N}"

# Retain only games where both home and away teams meet the minimum games requirement
n_before = game_view.height
game_view_filtered = game_view.filter(
    pl.col(f"home_{SENTINEL}").is_not_null()
    & pl.col(f"away_{SENTINEL}").is_not_null()
)
n_after   = game_view_filtered.height
n_dropped = n_before - n_after

# Log the impact of filtering on the dataset size
log.info(f"Rows before filtering    : {n_before:,}")
log.info(
    f"Rows dropped (< {MIN_GAMES_REQUIRED} preceding games "
    f"for at least one team): {n_dropped:,} ({n_dropped / n_before:.1%})"
)
log.info(f"Rows after filtering     : {n_after:,}")

### 12. Train / Validation / Test Temporal Split

* **What:** Splits the feature dataset chronologically into training, validation, and testing periods.
* **Why:** To evaluate models on future seasons rather than randomly mixed historical observations.
* **Reasoning/Choices:** Temporal splitting better reflects real forecasting conditions and reduces leakage from season-level trends or repeated team contexts.

Split by season (**no shuffle**) to respect temporal causality.
No game from a subsequent season can appear in a previous set.

| Set | Seasons | Rationale |
| --- | --- | --- |
| **Train** | 2000-01 → 2018-19 | ~19 seasons to train the model |
| **Val** | 2019-20 → 2021-22 | 3 recent seasons for hyperparameter tuning |
| **Test** | 2022-23 → 2025-26 | 4 seasons for final hold-out evaluation |

In [ ]:
# Define temporal boundaries for data splitting
TRAIN_END  = config["split"]["train_end"]
VAL_START  = config["split"]["val_start"]
VAL_END    = config["split"]["val_end"]
TEST_START = config["split"]["test_start"]

# Perform temporal splitting based on season strings
train_df = game_view_filtered.filter(pl.col("season") <= TRAIN_END)
val_df   = game_view_filtered.filter(
    (pl.col("season") >= VAL_START) & (pl.col("season") <= VAL_END)
)
test_df  = game_view_filtered.filter(pl.col("season") >= TEST_START)

# Sanity checks to ensure no data leakage or overlap across sets
assert train_df.height + val_df.height + test_df.height == game_view_filtered.height, \
    "Split is not exhaustive — check seasonal boundaries."
assert str(train_df["season"].max()) < str(val_df["season"].min()), \
    "Train/Val overlap detected!"
assert str(val_df["season"].max()) < str(test_df["season"].min()), \
    "Val/Test overlap detected!"

# Log the distribution of the splits
total = game_view_filtered.height
for name, split in [("Train", train_df), ("Val  ", val_df), ("Test ", test_df)]:
    pct = split.height / total
    log.info(
        f"{name}: {split.height:,} games ({pct:.1%}) | "
        f"seasons {split['season'].min()} – {split['season'].max()}"
    )

### 13. Saving

* **What:** Persists the final feature dataset and logs its shape, column set, and target distributions.
* **Why:** To provide a stable handoff artifact for model training and comparison.
* **Reasoning/Choices:** Saving only after validation and temporal splitting ensures downstream notebooks consume a documented, reproducible, model-ready dataset.

In [ ]:
# Assign a 'split' label to each game based on the previously defined temporal boundaries
game_view_final = game_view_filtered.with_columns(
    pl.when(pl.col("season") <= TRAIN_END).then(pl.lit("train"))
      .when(pl.col("season") <= VAL_END).then(pl.lit("val"))
      .otherwise(pl.lit("test"))
      .alias("split")
)

# Write the final processed and split dataset to CSV
game_view_final.write_csv(OUT_FILE)

# Log final status and dataset summary
log.info(f"File saved    : {OUT_FILE}")
log.info(f"Final shape   : {game_view_final.height:,} rows × {game_view_final.width} columns")

# Summary of split distribution
counts = {s: game_view_final.filter(pl.col("split") == s).height for s in ["train", "val", "test"]}
log.info(f"Split → train: {counts['train']:,} | val: {counts['val']:,} | test: {counts['test']:,}")

# List all columns in the final dataset
log.info(f"\nColumns in final dataset ({game_view_final.width}):")
for i, c in enumerate(game_view_final.columns, 1):
    log.info(f"  {i:2d}. {c}")